Here we will build models by putting together different models. For this, we will simply put together a vision encoder and an RNN. 

The vision encoder is supposed to learn or have learnt the internal representations and with the RNN we want to see how those internal representations are evolved over time: think mental rotation, visual memory. 


Training can be either end to end (training both the encoder and rnn) or we can freeze the encoder and just train the RNN (the dynamics). We do need data from different views (think: we are rotating the object)

Use an optic flow model to simulate what the pixel values will be as the object rotates and then fit those to the model. 

In this example, what we are building is essentially a perception → state → dynamics pipeline. The vision encoder is turning images (pixel level intensities) into internal representations and the RNN is updating those representations over time. Think of it this way: The vision model tells you what you’re seeing.
The RNN tells you how that internal state evolves when nothing new is shown.


**Conceptual Blocks**

- a Frozen or trainable vision encoder
- RNN on top of embeddings
- A very simple readout head which will be task dependent

so in simple terms: 

$z_t = f(x_t)$

$h_t = rnn(z_t, h_{t-1})$

$y_t = readout(h_t)$

In [ ]:
import torch
import torch.nn as nn

class VisionRNN(nn.Module):
    def __init__(self, encoder, embed_dim, hidden_dim):
        super().__init__()

        self.encodeer = encoder
        self.rnn = nn.GRU(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.readout = nn.Linear(hidden_dim, 1) # this is task specific

    def forward(self, x_seq):

        # x_seq is a sequence and let's assume that the dimensions are:
        # B, T, C, H, W with B being the batch size, T length of the sequence, C number of channels, H height, and W width
        B, T, C, H, W = x_seq.shape
        # what is this/
        x_seq = x_seq.view(B*T, C, H, W)

        z = self.encoder(x_seq)            # [B*T, emb_dim]
        z = z.view(B, T, -1)

        h, _ = self.rnn(z)                 # [B, T, hidden_dim]
        y = self.readout(h)                # [B, T, output_dim]

        return
